# 01 · Object detection — interactive walkthrough

A hands-on companion to the [report](report.qmd). Every model adapter returns a
`supervision.Detections`, so the same few lines annotate any model's output.

Requires the data + detection dependency groups, run from the repo root:

```bash
uv sync --group data --group detection
uv run jupyter lab   # then open modules/01-object-detection/explore.ipynb
```

In [ ]:
import matplotlib.pyplot as plt
from cvkit.devices import get_device, describe_device
from cvkit.io import load_image
from cvkit.viz import annotate_detections
import detect

device = get_device()
print(describe_device(device))

## Grab a few COCO images
Downloads (and caches) a small COCO val slice via the FiftyOne zoo.

In [ ]:
from cvkit.data import load_coco_val_subset

ds = load_coco_val_subset(max_samples=8)
paths = [s.filepath for s in ds][:4]
img = load_image(paths[0])
plt.imshow(img); plt.axis('off'); plt.show()

## Run a closed-set detector (YOLO26)
Predicts one of the 80 COCO classes per box. `data['class_name']` carries the labels.

In [ ]:
model = detect.build_detector('yolo26n', device)
det = model.predict(img, conf=0.3)
labels = [f"{n} {c:.2f}" for n, c in zip(det.data.get('class_name', []), det.confidence)]
plt.figure(figsize=(8, 8))
plt.imshow(annotate_detections(img, det, labels=labels)); plt.axis('off'); plt.show()
print(list(det.data.get('class_name', [])))

## Compare two models on the same image
Same image, same code path — only the adapter key changes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, key in zip(axes, ['yolo26n', 'rfdetr-nano']):
    m = detect.build_detector(key, device)
    d = m.predict(img, conf=0.3)
    lbls = [f"{n} {c:.2f}" for n, c in zip(d.data.get('class_name', []), d.confidence)]
    ax.imshow(annotate_detections(img, d, labels=lbls))
    ax.set_title(m.meta.display); ax.axis('off')
plt.show()

## Open-vocabulary: detect anything you can name
Grounding DINO takes free text — try concepts that aren't in COCO's 80 classes.

In [ ]:
gd = detect.build_open_vocab(device)
det = gd.detect(img, ['plate', 'napkin', 'logo', 'hand', 'cup'])
lbls = [f"{n} {c:.2f}" for n, c in zip(det.data.get('class_name', []), det.confidence)]
plt.figure(figsize=(8, 8))
plt.imshow(annotate_detections(img, det, labels=lbls)); plt.axis('off'); plt.show()
print('out-of-COCO:', sorted({n for n in det.data.get('class_name', []) if n.lower() not in detect.NAME_TO_IDX}))

## Next
Run the full benchmark with `uv run python run.py` from this folder, or read the
[report](report.qmd) for the rigorous numbers.